# Mercor Cheating Detection: Cost-Aware Ensembling Pipeline

This notebook contains the complete training and inference pipeline for the Mercor Cheating Detection task.

### Approach Overview
This solution frames the cheating detection task as a **cost-optimization problem** rather than standard classification. Because the cost of False Negatives (missed cheaters: $600) heavily outweighs False Positives (manual review: $150), the pipeline is engineered to produce exceptionally well-calibrated probabilities, allowing an optimizer to select optimal `t_pass` and `t_block` thresholds.

**Key Components:**
1. **Behavioral Feature Engineering:** Deriving intensity ratios, time-flag distributions, and Mutual Information (MI) guided interaction pairs from raw behavioral data.
2. **Social Graph Analysis:** Extracting 1-hop and 2-hop cheating densities across a 1.7M-edge graph using vectorized sparse matrix operations, alongside Louvain community detection.
3. **Cost-Calibrated Ensemble:** A 3-layer architecture utilizing XGBoost, LightGBM, and CatBoost. Each model undergoes 5-fold Stratified OOF training with strict Isotonic Calibration to fix tree overconfidence.
4. **Logistic Stacking:** Blending calibrated probabilities to achieve minimum theoretical cost.

*(Note: The codebase is modularized and imported via the attached `cheating-codebase` dataset.)*

In [ ]:
import os
import zipfile
import shutil
import warnings
warnings.filterwarnings('ignore')

# --- Environment Setup ---
# Find where Kaggle mounted the data recursively
import glob
data_dirs = glob.glob('/kaggle/input/**/train.csv', recursive=True)
code_dirs = glob.glob('/kaggle/input/**/train_pipeline.py', recursive=True)
code_zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)

DATA_DIR = os.path.dirname(data_dirs[0]) if data_dirs else "/kaggle/input/mercor-cheating-detection/"
WORK_DIR = "/kaggle/working/cheating"
os.makedirs(f"{WORK_DIR}/data", exist_ok=True)

# Unpack modular codebase into the working directory
if code_dirs:
    # Kaggle automatically unzipped the dataset. 
    # train_pipeline.py is in 'scripts/', so its parent's parent is the root
    CODE_ROOT = os.path.dirname(os.path.dirname(code_dirs[0]))
    print(f"Found unzipped codebase at: {CODE_ROOT}")
    os.system(f"cp -r {CODE_ROOT}/* {WORK_DIR}/")
elif code_zips:
    # Found a zip file
    print(f"Found zipped codebase at: {code_zips[0]}")
    with zipfile.ZipFile(code_zips[0], 'r') as zip_ref:
        zip_ref.extractall(WORK_DIR)
else:
    print("WARNING: Could not find the codebase. Did you attach the cheating-codebase dataset?")

# Link competition data
for filename in ['train.csv', 'test.csv', 'feature_metadata.json']:
    src = os.path.join(DATA_DIR, filename)
    dst = os.path.join(WORK_DIR, 'data', filename)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f"Linked {filename}")

### 1. Install Dependencies
Ensuring optimized versions of `networkx`, `scipy` (for sparse matrix ops), and tree libraries.

In [ ]:
!cd {WORK_DIR} && pip install -r requirements.txt -q

### 2. Execute Training Pipeline
This step executes the main pipeline script. It will:
- Process 112,000+ labeled rows
- Generate 64 behavioral features and graph embeddings
- Train OOF base learners
- Optimize the cost metric directly to find the ideal thresholds
- Output `submission.csv`

In [ ]:
# Run the full pipeline (graph computation + ensemble stacking)
!cd {WORK_DIR} && python3 scripts/train_pipeline.py --no-pseudo

### 3. Finalize Submission
Moving the generated predictions into the root directory for Kaggle's evaluation engine.

In [ ]:
import shutil
import os

WORK_DIR = "/kaggle/working/cheating"
submission_path = f"{WORK_DIR}/outputs/submissions/submission.csv"

if os.path.exists(submission_path):
    shutil.copy(submission_path, "/kaggle/working/submission.csv")
    print("Submission successfully prepared at: /kaggle/working/submission.csv")
else:
    print("Error: submission.csv not found. Make sure the training pipeline completed successfully.")